# Wanderbricks Bookings & Users

**Dataset:** `samples.wanderbricks.bookings`, `samples.wanderbricks.users`

**Difficulty:** Medium

**Topics:** join, aggregation, anti-join, window

In [0]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window as W

bookings = spark.read.table("samples.wanderbricks.bookings")
users = spark.read.table("samples.wanderbricks.users")

## Problem 1

Join bookings with users on `user_id`. Compute total revenue and booking count per user country.

**Expected output columns:**
- `country`
- `booking_count`
- `total_revenue`

In [0]:
# Problem 1 - write your solution here
# Assign your result to: result_1

result_1 = (
    bookings
    .join(users, "user_id")
    .groupBy("country")
    .agg(
        F.count("booking_id").alias("booking_count"),
        F.sum("total_amount").alias("total_revenue")
    )
)
display(result_1)

In [0]:
import pandas as pd

In [0]:
bookings_pd = bookings.toPandas()
users_pd = users.toPandas()

In [0]:
bookings_pd = bookings_pd.astype({
    'check_in':'string',
    'check_out':'string',
    'status':'string',
    'check_in':'datetime64[ns]',
    'check_out':'datetime64[ns]'
})

users_pd = users_pd.astype({
    'email': 'string',
    'name': 'string',
    'country': 'string',
    'user_type': 'string',
    'company_name': 'string'
})

In [0]:
result_1_pd = (
    pd
    .merge(bookings_pd, users_pd, on="user_id")
    .groupby("country")
    .agg(
        booking_count = ('booking_id', 'count'),
        total_revenue = ('total_amount', 'sum')
    )
    .reset_index()
)

display(result_1_pd)

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'country' in cols, "Missing column: country"
assert 'booking_count' in cols, "Missing column: booking_count"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_rev = result_1.agg(F.min('total_revenue')).collect()[0][0]
assert float(min_rev) >= 0, f"Expected total_revenue >= 0, found min={min_rev}"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

Find the top 10 most active users by number of bookings.

**Expected output columns:**
- `user_id`
- `name`
- `country`
- `booking_count`

In [0]:
# Problem 2 - write your solution here
# Assign your result to: result_2

result_2 =  (
    bookings
    .groupBy("user_id")
    .agg(
        F.count("booking_id").alias("booking_count")
    )
    .join(users, "user_id")
    .select("user_id", "name", "country", "booking_count")
    .orderBy(F.col("booking_count").desc())
    .limit(10)
)

result_2.display()

In [0]:
result_2_pd = (
    bookings_pd
    .groupby("user_id")
    .agg(
        booking_count = ("booking_id", "count")
    )
    .reset_index()
    .merge(users_pd[['user_id', 'name', 'country']], on='user_id')
    .sort_values("booking_count", ascending=False)
    .head(10)
)

display(result_2_pd)

In [0]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'user_id' in cols, "Missing column: user_id"
assert 'name' in cols, "Missing column: name"
assert 'country' in cols, "Missing column: country"
assert 'booking_count' in cols, "Missing column: booking_count"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert cnt <= 10, f"Expected at most 10 rows (top 10), got {cnt}"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

Find users who have never made a booking using a left anti-join.

**Expected output columns:**
- `user_id`
- `name`
- `country`
- `user_type`

In [0]:
# Problem 3 - write your solution here
# Assign your result to: result_3

result_3 = (
    users
    .join(bookings, 'user_id', 'left_anti')
    .select("user_id", "name", "country", "user_type")
)

display(result_3)

In [0]:
result_3_pd = (
    users_pd[~users_pd['user_id'].isin(bookings_pd['user_id'])]
)

display(result_3)

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'user_id' in cols, "Missing column: user_id"
assert 'name' in cols, "Missing column: name"
assert 'country' in cols, "Missing column: country"
assert 'user_type' in cols, "Missing column: user_type"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt >= 0, f"Expected rows >= 0, got {cnt}"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

Compare booking patterns between business users (`is_business = true`) and personal users. Compute average guests, average nights, average amount, and booking count per segment.
Use `datediff(check_out, check_in)` for nights.

**Expected output columns:**
- `is_business`
- `avg_guests`
- `avg_nights`
- `avg_amount`
- `booking_count`

In [0]:
# Problem 4 - write your solution here
# Assign your result to: result_4

result_4 = (
    bookings
    .join(users, 'user_id')
    .groupBy('is_business')
    .agg(
        F.avg('guests_count').alias('avg_guests'),
        F.avg(F.datediff('check_out', 'check_in')).alias('avg_nights'),
        F.avg('total_amount').alias('avg_amount'),
        F.count('booking_id').alias('booking_count')
    )
)

display(result_4)

In [0]:
bookings_pd['nights'] = (bookings_pd['check_out'] - bookings_pd['check_in']).dt.days

result_4_pd = (
    bookings_pd
    .merge(users_pd, on='user_id')
    .groupby('is_business')
    .agg(
        avg_guests = ('guests_count', 'mean'),
        avg_nights = ('nights', 'mean'),
        avg_amount = ('total_amount', 'mean'),
        booking_count = ('booking_id', 'count')
    )
    .reset_index()
    .sort_values('is_business', ascending=True)
)

display(result_4_pd)

In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'is_business' in cols, "Missing column: is_business"
assert 'avg_guests' in cols, "Missing column: avg_guests"
assert 'avg_nights' in cols, "Missing column: avg_nights"
assert 'avg_amount' in cols, "Missing column: avg_amount"
assert 'booking_count' in cols, "Missing column: booking_count"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert cnt <= 2, f"Expected at most 2 rows (true/false), got {cnt}"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

Find users with at least one cancelled booking (`status = 'cancelled'`).

**Expected output columns:**
- `user_id`
- `name`
- `country`
- `cancelled_bookings`

In [0]:
# Problem 5 - write your solution here
# Assign your result to: result_5

c_bookings = (
    bookings
    .filter(F.col('status') == 'cancelled')
    .groupBy('user_id')
    .agg(F.count('booking_id').alias('cancelled_bookings'))
)
 
result_5 = (
    users
    .join(c_bookings, 'user_id')
    .select('user_id', 'name', 'country', 'cancelled_bookings')
    .orderBy(F.col('cancelled_bookings').desc())
)

result_5.display()

In [0]:
result_5_pd = (
    bookings_pd[bookings_pd['status']=='cancelled']
    .groupby('user_id')
    .agg(
        cancelled_bookings = ('booking_id', 'count')
    )
    .reset_index()
    .merge(users_pd[['user_id', 'name', 'country']], on='user_id')
    .sort_values('cancelled_bookings', ascending=False)
)

result_5_pd.display()

In [0]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'user_id' in cols, "Missing column: user_id"
assert 'name' in cols, "Missing column: name"
assert 'country' in cols, "Missing column: country"
assert 'cancelled_bookings' in cols, "Missing column: cancelled_bookings"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt >= 0, f"Expected rows >= 0, got {cnt}"
if cnt > 0:
    min_cancelled = result_5.agg(F.min('cancelled_bookings')).collect()[0][0]
    assert min_cancelled >= 1, f"Expected cancelled_bookings >= 1, found min={min_cancelled}"
print(f"Problem 5 passed ✓  ({cnt} rows)")

## Problem 6

Using a window function, rank users by total spend within each country. Keep only rank <= 3.

**Expected output columns:**
- `country`
- `user_id`
- `name`
- `total_spend`
- `rank`

In [0]:
# Problem 6 - write your solution here
# Assign your result to: result_6
w = W.partitionBy('country').orderBy(F.col('total_spend').desc())
result_6 = (
    bookings
    .join(users, 'user_id')
    .groupBy('country', 'user_id', 'name')
    .agg(
        F.sum('total_amount').alias('total_spend')
    )
    .withColumn(
        'rank',
        F.rank().over(w)
    )
    .filter(F.col('rank') <= 3)
    .orderBy('country', 'rank')
)

result_6.display()

In [0]:
result_6_pd = (
    bookings_pd
    .merge(users_pd, on='user_id')
    .groupby(['country', 'user_id', 'name'], as_index=False)
    .agg(
        total_spend = ('total_amount', 'sum')
    )
    .assign(rank=lambda df: df.groupby('country')['total_spend']
        .rank(method='min', ascending=False))
    .query('rank <= 3')
    .sort_values(['country', 'rank'])
)

result_6_pd.display()

In [0]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None - did you assign your DataFrame?"
assert hasattr(result_6, 'columns'), "result_6 must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'country' in cols, "Missing column: country"
assert 'user_id' in cols, "Missing column: user_id"
assert 'name' in cols, "Missing column: name"
assert 'total_spend' in cols, "Missing column: total_spend"
assert 'rank' in cols, "Missing column: rank"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_6.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
max_rank = result_6.agg(F.max('rank')).collect()[0][0]
assert max_rank <= 3, f"Expected rank <= 3, found max={max_rank}"
min_rank = result_6.agg(F.min('rank')).collect()[0][0]
assert min_rank >= 1, f"Expected rank >= 1, found min={min_rank}"
print(f"Problem 6 passed ✓  ({cnt} rows)")

## Problem 7

Calculate the average booking lead time in days (days between `created_at` and `check_in`) per user type.

**Expected output columns:**
- `user_type`
- `avg_lead_time_days`
- `booking_count`

In [0]:
# Problem 7 - write your solution here
# Assign your result to: result_7

result_7 = (
    bookings
    .withColumn(
        'lead_time',
        F.datediff('check_in', 'created_at')
    )
    .join(users, 'user_id')
    .groupBy('user_type')
    .agg(
        F.avg('lead_time').alias('avg_lead_time_days'),
        F.count('booking_id').alias('booking_count')
    )
    .orderBy(F.col('booking_count').desc())
)

result_7.display()

In [0]:
result_7_pd = (
    bookings_pd
    .assign(lead_time = lambda df: (df['check_in'] - df['created_at']).dt.days)
    .merge(users_pd, on='user_id')
    .groupby('user_type', as_index=False)
    .agg(
        avg_lead_time_days = ('lead_time', 'mean'),
        booking_count = ('booking_id', 'count')
    )
    .sort_values('booking_count', ascending=False)
)

result_7_pd.display()

In [0]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None - did you assign your DataFrame?"
assert hasattr(result_7, 'columns'), "result_7 must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'user_type' in cols, "Missing column: user_type"
assert 'avg_lead_time_days' in cols, "Missing column: avg_lead_time_days"
assert 'booking_count' in cols, "Missing column: booking_count"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_7.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_bc = result_7.agg(F.min('booking_count')).collect()[0][0]
assert min_bc >= 1, f"Expected booking_count >= 1, found min={min_bc}"
print(f"Problem 7 passed ✓  ({cnt} rows)")